In [ ]:
"""
Math Verify library is built on SymPy. Based on what i read, it is not compatible with LLMs, especially GSM8K
"""


from math_verify import verify, parse
from sympy import sympify
import re

def wrap(text):
    num = re.findall(r"-?\d+\.?\d*", str(text))
    if not num:
        return parse("x = 0")
    return num[-1]
sympify("2+3") == sympify("5")



True

In [5]:
# example of sentence transformers from HuggingFace page
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [6]:
# From chatgpt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load HuggingFace embedding model
model = SentenceTransformer("Qwen/Qwen2.5-Coder-0.5B-Instruct")

# -----------------------------
# Example dataset
# -----------------------------
query = "The cat is sleeping on the sofa"

candidates = [
    "A cat is lying on the couch",
    "The dog is running in the park",
    "A feline is resting on a sofa",
]

# -----------------------------
# Embedding function
# -----------------------------
def get_similarity(a, b):
    emb = model.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

# -----------------------------
# ZERO-SHOT (no examples)
# -----------------------------
print("\n=== 0-SHOT SIMILARITY ===")
for c in candidates:
    score = get_similarity(query, c)
    print(f"{query} <-> {c}")
    print("Similarity:", round(score, 4))
    print()

# -----------------------------
# FEW-SHOT EXAMPLES
# -----------------------------
examples_3shot = [
    ("The cat is sleeping", "A cat is resting"),
    ("Dogs are barking", "A dog is making noise"),
    ("Bird is flying", "A bird is in the air"),
]

examples_5shot = examples_3shot + [
    ("Man is eating food", "A person is having a meal"),
    ("Car is moving fast", "Vehicle is driving quickly"),
]

def build_prompt(query, examples):
    prompt = ""
    for a, b in examples:
        prompt += f"Example:\n{a}\n{b}\n\n"
    prompt += f"Target:\n{query}"
    return prompt

# -----------------------------
# 3-SHOT
# -----------------------------
print("\n=== 3-SHOT SIMILARITY ===")
for c in candidates:
    q = build_prompt(query, examples_3shot)
    score = get_similarity(q, c)
    print(f"{c}")
    print("Similarity:", round(score, 4))
    print()

# -----------------------------
# 5-SHOT
# -----------------------------
print("\n=== 5-SHOT SIMILARITY ===")
for c in candidates:
    q = build_prompt(query, examples_5shot)
    score = get_similarity(q, c)
    print(f"{c}")
    print("Similarity:", round(score, 4))
    print()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


=== 0-SHOT SIMILARITY ===
The cat is sleeping on the sofa <-> A cat is lying on the couch
Similarity: 0.999

The cat is sleeping on the sofa <-> The dog is running in the park
Similarity: 0.9986

The cat is sleeping on the sofa <-> A feline is resting on a sofa
Similarity: 0.9984


=== 3-SHOT SIMILARITY ===
A cat is lying on the couch
Similarity: 0.9907

The dog is running in the park
Similarity: 0.9897

A feline is resting on a sofa
Similarity: 0.9887


=== 5-SHOT SIMILARITY ===
A cat is lying on the couch
Similarity: 0.9912

The dog is running in the park
Similarity: 0.9908

A feline is resting on a sofa
Similarity: 0.9894



In [ ]:
# Temprature
"""
Temperature is a parameter that controls the randomness of the output generated by a language model. 
It is used during the sampling process to determine how likely the model is to choose less probable tokens.
A higher temperature will result in lower probability, i.e more creative outputs. A lower temperature will result in higher probability, i.e more predictable outputs. 

"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ----------------------------
# Load model (change if needed)
# ----------------------------
model_name = "Qwen/Qwen2.5-0.5B-Instruct" 

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# ----------------------------
# Prompt
# ----------------------------
prompt = "Solve: There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?\n"

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
# ----------------------------
# Function to generate with temperature
# ----------------------------
def generate_with_temperature(temp):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,          # IMPORTANT for temperature
            temperature=temp,
            top_p=0.95,
            repetition_penalty=1.1
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# ----------------------------
# Run experiments
# ----------------------------
temperatures = [0.3, 0.7, 1.0, 1.3]

for t in temperatures:
    print("\n" + "="*50)
    print(f"TEMPERATURE = {t}")
    print("="*50)
    print(generate_with_temperature(t))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


TEMPERATURE = 0.3
Solve: There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?
To determine the total number of pieces of fruit, we need to calculate the number of oranges and the number of nectarines separately, and then sum these quantities.

First, let's calculate the number of oranges:
- Each crate contains 150 oranges.
- There are 12 crates.
\[
\text{Total number of oranges} = 12 \times 15

TEMPERATURE = 0.7
Solve: There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?
To find the total number of pieces of fruit, we need to calculate the number of oranges from the crates and the number of nectarines from the boxes separately, then add these two quantities together.

First, let's calculate the number of oranges:
- Each crate contains 150 oranges.
- There are 1

In [ ]:
# model is too small
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

prompt = """Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

def generate_with_temperature(temp):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=temp,
            top_p=0.95
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

for t in [0.3, 0.7, 1.0, 1.3]:
    print("\n" + "="*50)
    print("TEMPERATURE:", t)
    print("="*50)
    print(generate_with_temperature(t))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


TEMPERATURE: 0.3
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?
assistant
To solve the problem, we need to calculate the total number of oranges and nectarines separately and then sum them up.

First, let's calculate the total number of oranges:
- There are 12 crates.
- Each crate contains 150 oranges.
So, the total number of oranges is:
\[ 12 \text{ crates} \times 150 \text{ oranges/crate} = 1800 \text{ oranges} \]

Next,

TEMPERATURE: 0.7
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?
assistant
To solve this problem, we need to calculate the total number of oranges and nectarines separately and then sum them up.

1. Calculate

In [ ]:
# USING A BIGGER MODEL
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

prompt = """Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

def generate_with_temperature(temp):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=temp,
            top_p=0.95
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

for t in [0.3, 0.7, 1.0, 1.3]:
    print("\n" + "="*50)
    print("TEMPERATURE:", t)
    print("="*50)
    print(generate_with_temperature(t))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

c:\Users\param\CodingWorkspace\AlgorithmicSuperIntelligenceInternship\ASI-venv-311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\param\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]